# 08 · Metrics, dataset health and historical monitoring

Validation asks whether this batch passes. Monitoring asks how quality changes across runs. All trend data below is explicitly synthetic; current-fixture metrics are computed separately. An increasing defect rate deserves attention even before a hard gate fails.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Inspect the source
Each JSON line is one order envelope. Preserve the original text and source filename before parsing so even corrupt lines remain accountable.


In [ ]:
ORDER_FIELDS = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "status",
    "order_date",
    "shipped_date",
    "cancellation_reason",
    "order_total",
    "seller_id",
    "country",
    "arrival_date",
]
order_schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in ORDER_FIELDS]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)


def read_orders(path):
    # Preserve one source envelope per physical JSON line, including malformed JSON.
    # Source row IDs are materialized before branching; raw_text supports replay.
    raw = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_text")
        .withColumn("source_file", F.input_file_name())
        .withColumn("source_row_id", F.monotonically_increasing_id())
    )
    parsed = raw.withColumn(
        "parsed",
        F.from_json(
            "raw_text",
            order_schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    return parsed.select("source_row_id", "source_file", "raw_text", "parsed.*").cache()


orders = read_orders(f"{RAW_PATH}/orders")
orders.count()  # materialize once before splitting
customers = spark.read.option("header", True).csv(f"{RAW_PATH}/customers")
products = spark.read.option("header", True).csv(f"{RAW_PATH}/products")
items = spark.read.option("header", True).csv(f"{RAW_PATH}/order_items")
orders.show(30, truncate=False)


## Compute current metrics
Count source occurrences, distinguish duplicate keys from duplicate excess rows, and state metric denominators. Decimal amounts are summarized statistically for observation, not financial reconciliation.


In [ ]:
typed = (
    orders.withColumn("customer_id_clean", F.trim("customer_id"))
    .withColumn("status_clean", F.upper(F.trim("status")))
    .withColumn("qty", F.expr("try_cast(quantity as int)"))
    .withColumn("price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("discount_value", F.expr("try_cast(discount as decimal(8,2))"))
    .withColumn("total", F.expr("try_cast(order_total as decimal(18,2))"))
    .withColumn("event_date", F.expr("try_cast(order_date as date)"))
    .withColumn("shipped_on", F.expr("try_cast(shipped_date as date)"))
    .withColumn("arrived_on", F.expr("try_cast(arrival_date as date)"))
)
# Reference tables are deduplicated for membership joins, not as a silent repair.
# A separate customer-key check still exposes duplicate reference records.
customer_keys = (
    customers.select(F.col("customer_id").alias("customer_id_clean"))
    .distinct()
    .withColumn("known_customer", F.lit(True))
)
product_keys = (
    products.select("product_id").distinct().withColumn("known_product", F.lit(True))
)
typed = (
    typed.join(customer_keys, "customer_id_clean", "left")
    .join(product_keys, "product_id", "left")
    .withColumn("key_count", F.count("*").over(Window.partitionBy("order_id")))
)

# A predicate means PASS. NULL is a failure unless the rule explicitly permits it.
rules = [
    (
        "DQ000",
        "parseable",
        "raw_text",
        "Malformed JSON",
        F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull(),
    ),
    (
        "DQ001",
        "customer present",
        "customer_id",
        "Missing or blank customer",
        F.length("customer_id_clean") > 0,
    ),
    (
        "DQ002",
        "positive integer quantity",
        "quantity",
        "Not an integer in 1..1000",
        F.col("qty").between(1, 1000),
    ),
    (
        "DQ003",
        "nonnegative price",
        "unit_price",
        "Invalid or negative price",
        F.col("price") >= 0,
    ),
    (
        "DQ004",
        "allowed status",
        "status",
        "Unknown status",
        F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED"),
    ),
    (
        "DQ005",
        "unique order key",
        "order_id",
        "Duplicate business key; quarantine all copies",
        F.col("key_count") == 1,
    ),
    (
        "DQ006",
        "event window",
        "order_date",
        "Invalid, future, old or outside daily window",
        F.col("event_date") == F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1),
    ),
    (
        "DQ007",
        "known customer",
        "customer_id",
        "Customer reference not found",
        F.coalesce(F.col("known_customer"), F.lit(False)),
    ),
    (
        "DQ008",
        "known product",
        "product_id",
        "Product reference not found",
        F.coalesce(F.col("known_product"), F.lit(False)),
    ),
    (
        "DQ009",
        "discount range",
        "discount",
        "Discount outside 0..100",
        F.col("discount_value").between(0, 100),
    ),
    (
        "BR001",
        "shipment date",
        "shipped_date",
        "SHIPPED requires valid shipped_date",
        (F.col("status_clean") != "SHIPPED") | F.col("shipped_on").isNotNull(),
    ),
    (
        "BR002",
        "cancellation reason",
        "cancellation_reason",
        "CANCELLED requires reason",
        (F.col("status_clean") != "CANCELLED")
        | (F.length(F.trim("cancellation_reason")) > 0),
    ),
    (
        "BR003",
        "nonnegative total",
        "order_total",
        "Invalid or negative order total",
        F.col("total") >= 0,
    ),
    (
        "BR004",
        "order arithmetic",
        "order_total",
        "Header differs from quantity times unit price",
        F.abs(F.col("total") - F.col("qty") * F.col("price"))
        <= F.lit("0.01").cast("decimal(18,2)"),
    ),
]
failure_structs = [
    F.when(
        ~F.coalesce(predicate, F.lit(False)),
        F.struct(
            F.lit(rule_id).alias("dq_rule_id"),
            F.lit(name).alias("dq_rule_name"),
            F.lit(column).alias("dq_column"),
            F.lit(reason).alias("dq_reason"),
        ),
    )
    for rule_id, name, column, reason, predicate in rules
]
scored = (
    typed.withColumn(
        "dq_failures", F.filter(F.array(*failure_structs), lambda x: x.isNotNull())
    )
    .withColumn(
        "dq_status", F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID")
    )
    .withColumn("pipeline_run_id", F.lit(RUN_ID))
    .withColumn("processing_timestamp", F.current_timestamp())
    .cache()
)
scored.count()
valid = scored.filter("dq_status = 'VALID'")
rejected = scored.filter("dq_status = 'INVALID'")
valid.select("order_id", "quantity", "status", "dq_status").show(truncate=False)
rejected.select("order_id", "source_row_id", "dq_failures").show(30, truncate=False)

row_count = scored.count()
null_count = scored.filter(
    "customer_id_clean is null or customer_id_clean = ''"
).count()
distinct_count = (
    scored.select("order_id").where("order_id is not null").distinct().count()
)
duplicate_count = scored.where("key_count > 1").select("order_id").distinct().count()
invalid_count = rejected.count()
ri_failed = scored.filter("known_customer is null or known_product is null").count()
metrics = dict(
    row_count=row_count,
    null_count=null_count,
    null_percentage=100 * null_count / row_count,
    distinct_count=distinct_count,
    duplicate_count=duplicate_count,
    invalid_count=invalid_count,
    invalid_percentage=100 * invalid_count / row_count,
    completeness=100 * (row_count - null_count) / row_count,
    uniqueness=100
    * scored.filter("key_count = 1 and order_id is not null").count()
    / row_count,
    validity=100 * (row_count - invalid_count) / row_count,
    referential_integrity=100 * (row_count - ri_failed) / row_count,
)
print(metrics)
scored.agg(F.min("price"), F.max("price"), F.mean("price"), F.stddev("price")).show()
scored.filter(F.col("event_date") <= F.to_date(F.lit(PROCESSING_DATE))).agg(
    F.datediff(F.to_date(F.lit(PROCESSING_DATE)), F.max("event_date")).alias(
        "freshness_days"
    )
).show()
print(
    "Future dates counted separately:",
    scored.filter(F.col("event_date") > F.to_date(F.lit(PROCESSING_DATE))).count(),
)


## Volume, categories, partitions and distributions
Individual good rows cannot prove dataset completeness. Compare expected partitions and previous volumes; use an agreed baseline. The z-score illustration excludes zero standard deviation and is only a simple heuristic.


In [ ]:
previous_volume, today_volume, max_change_pct = 100000, 12000, 30.0
change_pct = 100.0 * (today_volume - previous_volume) / previous_volume
print(
    "Volume change", change_pct, "FAIL" if abs(change_pct) > max_change_pct else "PASS"
)
expected_dates = spark.createDataFrame(
    [("2024-01-01",), ("2024-01-02",), ("2024-01-03",)], "order_date string"
)
expected_dates.join(
    orders.select("order_date").distinct(), "order_date", "left_anti"
).show()
scored.filter(
    ~F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED")
).select("order_id", "status_clean").show()
scored.groupBy("status_clean").count().withColumn(
    "share_pct", 100 * F.col("count") / row_count
).show()
baseline_mean, baseline_sd = 25.0, 5.0
scored.withColumn("baseline_z", (F.col("price") - baseline_mean) / baseline_sd).filter(
    F.abs("baseline_z") > 3
).select("order_id", "price", "baseline_z").show()


## Thirty synthetic runs: PASS can deteriorate
Store metrics in long form with dataset, rule, numerator, denominator, threshold and status. Run IDs make this simulation distinguishable from actual execution evidence. Query histories by simulation ID to keep reruns separate.


In [ ]:
from datetime import date, timedelta

simulation_id = "simulation_" + RUN_ID
audit_rows = []
for day in range(30):
    run = f"{simulation_id}_{day:02}"
    business_date = (date(2023, 12, 4) + timedelta(days=day)).isoformat()
    n = 10000
    null_pct = round(0.10 + day * 0.012, 3)
    values = [
        (
            "DQ001",
            "customer completeness",
            "null_percentage",
            null_pct,
            0.5,
            int(n * null_pct / 100),
        ),
        (
            "DQALL",
            "record validity",
            "reject_percentage",
            0.2 + day * 0.02,
            1.0,
            int(n * (0.2 + day * 0.02) / 100),
        ),
        ("DQ005", "unique keys", "duplicate_count", 0.0, 0.0, 0),
        ("VOL", "daily volume", "row_count", float(n), 12000.0, 0),
        ("REC", "row accounting", "reconciliation_difference", 0.0, 0.0, 0),
    ]
    for rule, name, metric, value, threshold, failed in values:
        audit_rows.append(
            (
                simulation_id,
                run,
                business_date,
                "orders",
                rule,
                name,
                metric,
                float(value),
                float(threshold),
                failed,
                n,
                100.0 * failed / n,
                "PASS" if value <= threshold else "FAIL",
            )
        )
history = spark.createDataFrame(
    audit_rows,
    "simulation_id string, run_id string, business_date string, dataset string, rule_id string, rule_name string, metric_name string, metric_value double, threshold double, failed_records long, total_records long, failure_percentage double, status string",
).withColumn("processing_timestamp", F.current_timestamp())
history.write.mode("errorifexists").parquet(f"{AUDIT_PATH}/dq_results/{simulation_id}")
history_read = spark.read.parquet(f"{AUDIT_PATH}/dq_results/{simulation_id}")
history_read.createOrReplaceTempView("dq_history")
spark.sql("""SELECT business_date, run_id,
    max(CASE WHEN metric_name='null_percentage' THEN metric_value END) AS null_pct,
    max(CASE WHEN metric_name='reject_percentage' THEN metric_value END) AS reject_pct,
    max(CASE WHEN metric_name='duplicate_count' THEN metric_value END) AS duplicates,
    max(CASE WHEN metric_name='row_count' THEN metric_value END) AS rows,
    max(CASE WHEN metric_name='reconciliation_difference' THEN metric_value END) AS reconciliation_diff
    FROM dq_history GROUP BY business_date,run_id ORDER BY business_date""").show(
    30, truncate=False
)
trend = history_read.filter("metric_name='null_percentage'").withColumn(
    "previous_value", F.lag("metric_value").over(Window.orderBy("business_date"))
)
trend.withColumn(
    "deteriorating", F.col("metric_value") > F.col("previous_value")
).select("business_date", "metric_value", "status", "deteriorating").show(30)


## Exercise
Design an alert for three consecutive increases, while keeping the hard 0.5% gate unchanged. Monitoring thresholds need seasonality, holiday and backfill context. A daily freshness maximum alone can be fooled by one recent record: also inspect age distribution and missing partitions.
